In [2]:
import os
from google.colab import files


# Upload your kaggle.json
if not os.path.exists('/content/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()

# Configure Kaggle directory
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API successfully configured!")


Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle API successfully configured!


In [19]:
!pip install -q spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 104.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
!pip install summa


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for summa: filename=summa-1.2.0-py3-none-any.whl size=54441 sha256=928d484a89231f3fb6ecc4e26d034da7104a943b0a6948e0331d1d9df94da6f7
  Stored in directory: /root/.cache/pip/wheels/75/be/7d/71f9113116fbdaa9ae7287a3017e9f1c6fbe95cadf170edbbd
Successfully built summa


In [4]:
# 1. Download the Judgment-Summary pair dataset (209 MB)
!kaggle datasets download -d rishipjain/indian-court-judgment-summary-pair
!unzip -q indian-court-judgment-summary-pair.zip -d ./summary_pairs

# 2. Download the ILDC Parquet dataset
!kaggle datasets download -d samridhishreya/ildc-single-1
!unzip -q ildc-single-1.zip -d ./ildc_dataset

print("All datasets downloaded and extracted successfully!")


Dataset URL: https://www.kaggle.com/datasets/rishipjain/indian-court-judgment-summary-pair
License(s): MIT
100% 55.2M/55.2M [00:04<00:00, 12.3MB/s]

Dataset URL: https://www.kaggle.com/datasets/samridhishreya/ildc-single-1
License(s): other
100% 86.0M/86.0M [00:05<00:00, 17.6MB/s]

All datasets downloaded and extracted successfully!


In [5]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load the Summarization CSV
df_summary = pd.read_csv("./summary_pairs/pair_dataset.csv")

# Clean the dataset (Drop missing values if any)
df_summary = df_summary.dropna(subset=['Judgment', 'Summary']).reset_index(drop=True)

# Rename the columns to match standard training formats
df_summary = df_summary.rename(columns={'Judgment': 'text', 'Summary': 'summary'})

# Convert into a Hugging Face Dataset and split into Train/Test
raw_dataset = Dataset.from_pandas(df_summary)
dataset_split = raw_dataset.train_test_split(test_size=0.1, seed=42)

print("Processed Summarization Dataset Structure:")
print(dataset_split)


Processed Summarization Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['ID', 'text', 'summary'],
        num_rows: 6247
    })
    test: Dataset({
        features: ['ID', 'text', 'summary'],
        num_rows: 695
    })
})


In [6]:
from summa import summarizer
from transformers import AutoTokenizer

model_name = "allenai/led-base-16384"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def extractive_prep(example):
    """
    Step 1: Extract 30% of the sentences using TextRank to shorten
    massive Indian legal cases down to essential arguments.
    """
    raw_text = example['text']
    extractive_summary = summarizer.summarize(raw_text, ratio=0.3)

    # Fallback to truncated raw text if TextRank returns nothing (due to short texts)
    if not extractive_summary:
        extractive_summary = raw_text[:8000]

    example['extractive_text'] = extractive_summary
    return example

def tokenize_function(examples):
    """
    Step 2: Tokenize the shortened text and the final target summaries.
    """
    # Tokenize input (Extractive text -> LED)
    model_inputs = tokenizer(
        examples["extractive_text"],
        max_length=4096,
        truncation=True
    )
    # Tokenize label (Target abstractive summary)
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=512,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Process a small subset (e.g., 500 samples) first to verify Colab training speeds
subset_train = dataset_split['train'].select(range(500))
subset_eval = dataset_split['test'].select(range(50))

# Apply the two-stage pipeline
train_dataset = subset_train.map(extractive_prep).map(tokenize_function, batched=True)
eval_dataset = subset_eval.map(extractive_prep).map(tokenize_function, batched=True)

print("Preprocessing complete! Ready for model training.")


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Preprocessing complete! Ready for model training.


In [7]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00


In [8]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=9e8c1ed6dabe4e1c87ec25aaf30f2828aeae8e8b77a2041ce18a7a83457d3470
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [9]:
!pip install rouge-score absl-py nltk


In [13]:
import torch
import numpy as np
import evaluate
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM

# 1. Load the pre-trained LED model
model_name = "allenai/led-base-16384"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2. Configure model global attention (Required for LED)
model.config.gradient_checkpointing = True
model.config.use_cache = False  # Set to False during training to save memory

# 3. Initialize evaluation metric
# (Kept in code, but we comment it out in the trainer to skip training-time ROUGE)
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# 4. Define data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 5. Define Training Arguments optimized for Colab T4 GPU (With Speed Optimizations)
training_args = Seq2SeqTrainingArguments(
    output_dir="./verdictlens_led_model",
    eval_strategy="steps",
    eval_steps=50,                  # ✅ CHANGED: Evaluate every 50 steps (very fast now)
    save_steps=50,                  # ✅ CHANGED: Save model checkpoint every 50 steps
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=False,    # 🔴 CRITICAL CHANGE: Disable generation to speed up training
    fp16=True,
    logging_steps=10,
    report_to="none"
)

# 6. Initialize Trainer (Bypassing slow training-time metrics computation)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    # compute_metrics=compute_metrics # 🔴 CRITICAL CHANGE: Commented out to prevent evaluation slow-downs
)

# 7. Start Training
print("🚀 Launching fast VerdictLens training...")
trainer.train()


Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

🚀 Launching fast VerdictLens training...


[transformers] Input ids are automatically padded from 1607 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1681 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 3311 to 4096 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 2938 to 3072 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 2469 to 3072 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 3264 to 4096 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1533 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 838 to 1024 to be a multiple of `config.attention_window`: 1024


Step,Training Loss,Validation Loss
50,20.839346,2.479094
100,19.108011,2.373484
150,17.007002,2.335995
189,18.444899,2.310633


[transformers] Input ids are automatically padded from 1450 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 920 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 941 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 666 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 739 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 874 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1103 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1036 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically pa

...

[transformers] Input ids are automatically padded from 2156 to 3072 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 2724 to 3072 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1503 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 984 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1391 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1110 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1255 to 2048 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 204 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=19.955224496346933, metrics={'train_runtime': 1045.0944, 'train_samples_per_second': 1.435, 'train_steps_per_second': 0.181, 'total_flos': 2108430158229504.0, 'train_loss': 19.955224496346933, 'epoch': 3.0})

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

model_dir = Path("./backend/ai_models/verdictlens_led_model")
model_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(model_dir))
tokenizer.save_pretrained(str(model_dir))

archive_path = shutil.make_archive(
    str(model_dir),
    "zip",
    root_dir=model_dir,
)
print(f"Model exported to {archive_path}")
files.download(archive_path)

In [14]:
import torch
import numpy as np
import pandas as pd
import evaluate
from tqdm.auto import tqdm

# 1. Set model to evaluation mode and move to GPU
model.eval()
model.to("cuda")

# 2. Reload ROUGE metric
rouge_metric = evaluate.load("rouge")

print("Generating summaries for the evaluation dataset...")
predictions = []
references = []

# 3. Generate summaries using batch processing
for batch in tqdm(eval_dataset):
    # Prepare inputs and move to GPU
    input_ids = torch.tensor([batch["input_ids"]]).to("cuda")
    attention_mask = torch.tensor([batch["attention_mask"]]).to("cuda")

    # Generate summary with Beam Search
    with torch.no_grad():
        summary_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=3,
            min_length=50,
            max_length=256,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    # Decode predictions and original summaries
    pred_str = tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    ref_str = tokenizer.decode(batch["labels"], skip_special_tokens=True, clean_up_tokenization_spaces=True)

    predictions.append(pred_str)
    references.append(ref_str)

# 4. Calculate Final ROUGE Metrics
scores = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=True)

print("\n📊 --- FINAL ROUGE METRICS ---")
for metric, score in scores.items():
    print(f"✨ {metric.upper()}: {round(score * 100, 4)}%")


Generating summaries for the evaluation dataset...


  0%|          | 0/50 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



📊 --- FINAL ROUGE METRICS ---
✨ ROUGE1: 48.9333%
✨ ROUGE2: 23.3711%
✨ ROUGEL: 26.0095%
✨ ROUGELSUM: 26.0743%


In [16]:
!pip install -q pypdf


In [18]:
import io
import re
import torch
from google.colab import files
from pypdf import PdfReader
from summa import summarizer

def preprocess_text(text):
    """
    Cleans the raw legal text extracted from the PDF [1].
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)  # Remove extra spacing/tabs
    return text.strip()

def extract_text_from_pdf(pdf_bytes):
    """
    Extracts text from a binary PDF file stream [1].
    """
    reader = PdfReader(io.BytesIO(pdf_bytes))
    full_text = []

    # Loop through pages and extract text
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text.append(text)

    return "\n".join(full_text)

def summarize_uploaded_pdf():
    """
    Interactive function: Prompts user to upload a PDF,
    extracts text, and prints the generated model summary [1].
    """
    print("📁 Please upload your Indian Court Judgment PDF:")
    uploaded = files.upload()

    # Get the filename and file content
    for file_name, file_content in uploaded.items():
        print(f"\n⚙️ Processing '{file_name}'...")

        # 1. Extract text from PDF
        raw_judgment_text = extract_text_from_pdf(file_content)
        if not raw_judgment_text.strip():
            print("❌ Could not extract text from this PDF. Please ensure it is a text-based (not scanned) PDF.")
            return

        print(f"📖 Extracted {len(raw_judgment_text.split())} words.")

        # 2. Run Preprocessing
        cleaned_text = preprocess_text(raw_judgment_text)

        # 3. Two-Stage: Extractive Summarization (TextRank)
        print("🧹 Running Extractive Summarization (TextRank)...")
        extractive_text = summarizer.summarize(cleaned_text, ratio=0.3)
        if not extractive_text:
            extractive_text = cleaned_text[:8000]

        # 4. Two-Stage: Tokenize and Abstractive Generation (LED)
        print("🧠 Generating Abstractive Summary (Your Fine-Tuned LED)...")
        inputs = tokenizer(extractive_text, max_length=4096, truncation=True, return_tensors="pt").to("cuda")

        with torch.no_grad():
            summary_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                num_beams=3,
                min_length=50,
                max_length=256,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        final_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

        print("\n" + "="*50)
        print(f"📝 --- GENERATED SUMMARY FOR {file_name.upper()} ---")
        print("="*50)
        print(final_summary)
        print("="*50)

# Run the function
summarize_uploaded_pdf()


📁 Please upload your Indian Court Judgment PDF:


Saving 60181_1989_1_1501_61910_Judgement_20-May-2025.pdf to 60181_1989_1_1501_61910_Judgement_20-May-2025 (1).pdf

⚙️ Processing '60181_1989_1_1501_61910_Judgement_20-May-2025 (1).pdf'...
📖 Extracted 13305 words.
🧹 Running Extractive Summarization (TextRank)...
🧠 Generating Abstractive Summary (Your Fine-Tuned LED)...

📝 --- GENERATED SUMMARY FOR 60181_1989_1_1501_61910_JUDGEMENT_20-MAY-2025 (1).PDF ---
The High Court of Delhi had approved the recommendation of the Shetty Commission that the recruitment to the Higher Judicial Service i.e. the District Judge Cadre from amongst the advocates should be 25% and appointment on the basis of merit should be 75% as determined by this Court in the case of All India Judges’ Association & Ors. v. State of Haryana. The High Court had also recommended that the qualifying service as a Civil Judge (Junior Division) should be not less than 5 years. In pursuance of the order dated 20 th April 4 2010 passed in the High Court	 the High Courts had amended

In [20]:
import io
import re
import spacy
import torch
from google.colab import files
from pypdf import PdfReader
from summa import summarizer

# Load SpaCy lightweight NLU engine
nlp = spacy.load("en_core_web_sm")

def extract_nlu_legal_entities(text):
    """
    NLU Component: Extracts critical legal entities, statutory citations,
    and court metadata to ensure key facts are preserved.
    """
    nlu_data = {
        "acts_sections": [],
        "dates": [],
        "parties": []
    }

    # 1. Rule-based NLU for Indian Legal Acts & Sections (IPC, CrPC, Constitution, etc.)
    section_pattern = r"(?:Section\s+\d+[A-Za-z]*(?:\s*\([0-9a-zA-Z]+\))*|\bIPC\b|\bCrPC\b|\bCPC\b|Article\s+\d+)"
    acts_found = list(set(re.findall(section_pattern, text, flags=re.IGNORECASE)))
    nlu_data["acts_sections"] = acts_found[:8]  # Capture key statutes

    # 2. Statistical NLU (SpaCy NER) for Dates and Organizations/Persons
    doc = nlp(text[:10000])  # Analyze the primary case facts and introductory clauses
    for ent in doc.ents:
        if ent.label_ == "DATE" and ent.text not in nlu_data["dates"]:
            nlu_data["dates"].append(ent.text)
        elif ent.label_ in ["PERSON", "ORG"] and len(ent.text) > 3:
            if ent.text not in nlu_data["parties"]:
                nlu_data["parties"].append(ent.text)

    nlu_data["dates"] = nlu_data["dates"][:4]
    nlu_data["parties"] = nlu_data["parties"][:6]

    return nlu_data

def format_nlu_header(nlu_data):
    """
    Formats extracted NLU entities into a factual anchor prefix.
    """
    header = "[CASE METADATA]\n"
    if nlu_data["acts_sections"]:
        header += f"Statutes/Provisions: {', '.join(nlu_data['acts_sections'])}\n"
    if nlu_data["parties"]:
        header += f"Parties/Entities: {', '.join(nlu_data['parties'])}\n"
    if nlu_data["dates"]:
        header += f"Key Dates: {', '.join(nlu_data['dates'])}\n"
    header += "[ARGUMENTS & EVIDENCE]\n"
    return header

def summarize_pdf_with_nlu():
    """
    Interactive function: Uploads PDF, applies NLU fact extraction,
    and generates an accurate, hallucination-resistant summary.
    """
    print("📁 Please upload your Indian Court Judgment PDF:")
    uploaded = files.upload()

    for file_name, file_content in uploaded.items():
        print(f"\n⚙️ Analyzing '{file_name}' with NLU...")

        # 1. Extract text from PDF
        reader = PdfReader(io.BytesIO(file_content))
        raw_text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

        if not raw_text.strip():
            print("❌ Could not extract text from this PDF.")
            return

        print(f"📖 Extracted {len(raw_text.split())} words.")

        # 2. Extract NLU Legal Entities (Fact Preservation)
        print("🔍 Extracting legal entities and statutory references (NLU)...")
        entities = extract_nlu_legal_entities(raw_text)
        nlu_header = format_nlu_header(entities)

        # 3. Extractive Condensation (TextRank)
        print("🧹 Running Extractive Summarization...")
        cleaned = re.sub(r'\s+', ' ', raw_text).strip()
        condensed_text = summarizer.summarize(cleaned, ratio=0.25) or cleaned[:6000]

        # 4. Combine NLU Metadata with Case Facts
        augmented_input = nlu_header + condensed_text

        # 5. Generate Final Abstractive Summary via Fine-Tuned LED
        print("🧠 Generating Fact-Preserved Abstractive Summary...")
        inputs = tokenizer(augmented_input, max_length=4096, truncation=True, return_tensors="pt").to("cuda")

        with torch.no_grad():
            summary_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                num_beams=3,
                min_length=60,
                max_length=280,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        final_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

        # 6. Display Output with Structured Fact Verification
        print("\n" + "="*60)
        print(f"🏛️ VERDICTLENS NLU SUMMARY: {file_name.upper()}")
        print("="*60)
        print("📌 Extracted Critical Entities:")
        print(f" • Acts/Sections: {', '.join(entities['acts_sections']) or 'None detected'}")
        print(f" • Key Parties:   {', '.join(entities['parties']) or 'None detected'}")
        print(f" • Dates:         {', '.join(entities['dates']) or 'None detected'}")
        print("\n📝 Abstractive Summary:")
        print(final_summary)
        print("="*60)

# Run the NLU-enhanced summarizer
summarize_pdf_with_nlu()


📁 Please upload your Indian Court Judgment PDF:


Saving 60181_1989_1_1501_61910_Judgement_20-May-2025.pdf to 60181_1989_1_1501_61910_Judgement_20-May-2025 (2).pdf

⚙️ Analyzing '60181_1989_1_1501_61910_Judgement_20-May-2025 (2).pdf' with NLU...
📖 Extracted 13305 words.
🔍 Extracting legal entities and statutory references (NLU)...
🧹 Running Extractive Summarization...
🧠 Generating Fact-Preserved Abstractive Summary...

🏛️ VERDICTLENS NLU SUMMARY: 60181_1989_1_1501_61910_JUDGEMENT_20-MAY-2025 (2).PDF
📌 Extracted Critical Entities:
 • Acts/Sections: Article 233
 • Key Parties:   THE SUPREME COURT OF INDIA, ALL INDIA JUDGES ASSOCIATION, VERSUS 
 
UNION OF INDIA, DISCUSSION, ANALYSIS, Limited 
Departmental Competitive Examination
 • Dates:         2019, 72900, 73015, 2021

📝 Abstractive Summary:
The High Court had approved the recommendation of the Shetty Commission for providing 25% quota for promotion to Higher Judicial Service i.e. cadre of District Judges against the existing 65% quota of Civil Judge (Senior Division). The High Court 